# Qwen Image 2.1 on Kaggle (Wan2GP)

Single T4 setup. Heavy weights go in `/tmp`.

## 1. Install Wan2GP

In [ ]:
%cd /kaggle/working
import os
if not os.path.exists('/kaggle/working/Wan2GP'):
    !git clone https://github.com/deepbeepmeep/Wan2GP.git
%cd /kaggle/working/Wan2GP

## 2. Install requirements

In [ ]:
!pip install -r requirements.txt

## 3. Download heavy models to `/tmp`

Only the transformer and text encoder. VAE is small; let Wan2GP download it.

### Transformer

In [ ]:
from huggingface_hub import hf_hub_download

base = hf_hub_download(
    repo_id='DeepBeepMeep/Qwen_image_2',
    filename='qwen_image_21_7B_int8_convrot.safetensors',
    local_dir='/tmp/qwen_image_21'
)
print('Base:', base)
!ls -lh /tmp/qwen_image_21

### Text encoder

In [ ]:
encoder = hf_hub_download(
    repo_id='DeepBeepMeep/Ideogram4',
    filename='Qwen3-VL-8B-Instruct/Qwen3-VL-8B-Instruct_int8_convrot.safetensors',
    local_dir='/tmp/qwen_image_21'
)
print('Encoder:', encoder)
!du -sh /tmp/qwen_image_21

## 4. Launch Wan2GP

Profile 5 is the low-VRAM profile. Only one T4 will be used.

In [ ]:
%cd /kaggle/working/Wan2GP

!python wgp.py --share

## 5. Add the local files as a finetune

In Wan2GP: pick **Qwen Image 2.1 7B** → Finetune Creator / Editor → point these fields at the `/tmp` files.

### Main checkpoint
```
/tmp/qwen_image_21/qwen_image_21_7B_int8_convrot.safetensors
```

### Text encoder checkpoint
```
/tmp/qwen_image_21/Qwen3-VL-8B-Instruct/Qwen3-VL-8B-Instruct_int8_convrot.safetensors
```

Architecture: `qwen_image_21_7B`

Keep the default Qwen Image 2.1 VAE from the source model.

## 6. Settings

- Resolution: `1024x1024`
- Steps: `40`
- Guidance: `4`
- KV Cache: Off
- Batch: `1`

## 7. Test prompt

**Prompt**
```
A cute orange cat sitting on a wooden table beside a small cup of coffee, warm morning sunlight coming through a window, realistic photography.
```

**Negative**
```
blurry, low quality, watermark, text
```